```mermaid
%%{init: {'theme':'base', 'themeVariables': { 'primaryColor':'#9B59B6','primaryTextColor':'#fff','primaryBorderColor':'#7D3C98','lineColor':'#E74C3C','secondaryColor':'#3498DB','tertiaryColor':'#F39C12'}}}%%
graph TB
    subgraph "RAGTool Workflow - Retrieval-Augmented Generation"
        A[🎓 Student Question] -->|Natural Language| B[🔢 Embedding Model]
        B -->|Query Vector| C[🔍 Vector Search]
        C -->|Retrieve| D[📚 Knowledge Base]
        D -->|Top-k Documents| E[📝 Context Assembly]
        E -->|Context + Question| F[🤖 OpenAI GPT]
        F -->|Inference| G[✨ Intelligent Answer]
        
        style A fill:#9B59B6,stroke:#7D3C98,stroke-width:3px,color:#fff
        style B fill:#E74C3C,stroke:#C0392B,stroke-width:3px,color:#fff
        style C fill:#3498DB,stroke:#2874A6,stroke-width:3px,color:#fff
        style D fill:#27AE60,stroke:#1E8449,stroke-width:3px,color:#fff
        style E fill:#F39C12,stroke:#D68910,stroke-width:3px,color:#fff
        style F fill:#E67E22,stroke:#CA6F1E,stroke-width:3px,color:#fff
        style G fill:#16A085,stroke:#117A65,stroke-width:3px,color:#fff
    end
    
    subgraph "RAG Benefits"
        H[🎯 Factual Accuracy]
        I[📊 Source Attribution]
        J[🔄 Current Information]
        K[🛡️ Reduced Hallucination]
        
        style H fill:#8E44AD,stroke:#6C3483,stroke-width:2px,color:#fff
        style I fill:#8E44AD,stroke:#6C3483,stroke-width:2px,color:#fff
        style J fill:#8E44AD,stroke:#6C3483,stroke-width:2px,color:#fff
        style K fill:#8E44AD,stroke:#6C3483,stroke-width:2px,color:#fff
    end
```

# 🤖 RAGTool - Retrieval-Augmented Generation

## 🎯 Learning Objectives

In this notebook, you will learn:
- ✅ How to implement **RAG (Retrieval-Augmented Generation)** with OpenSearch
- ✅ How to combine semantic search with **OpenAI GPT models**
- ✅ How to build agents that answer questions based on **your data**
- ✅ How to reduce AI hallucinations with **grounded responses**

## 📖 What is RAG?

**Retrieval-Augmented Generation (RAG)** is a technique that enhances LLM responses by:
1. 🔍 **Retrieving** relevant documents from a knowledge base
2. 📝 **Augmenting** the LLM prompt with retrieved context
3. 🤖 **Generating** an answer based on both the question and context

### Why Use RAG?

| Without RAG | With RAG |
|-------------|----------|
| ❌ LLM has limited knowledge (training cutoff) | ✅ Access to current, domain-specific data |
| ❌ May hallucinate facts | ✅ Grounded in retrieved documents |
| ❌ No source attribution | ✅ Can cite sources |
| ❌ Generic responses | ✅ Tailored to your data |

### RAG Workflow:
```
User Query → Vectorize → Semantic Search → Retrieve Docs → LLM (Query + Context) → Answer
```

---

In [ ]:
# Import required libraries
import sys
import json
import time
sys.path.append('..')

from agent_helpers import (
    get_os_client,
    configure_cluster_for_openai,
    create_openai_connector,
    register_and_deploy_openai_model,
    wait_for_model_deployment
)

print("✅ Libraries imported successfully")

## 🔧 Step 1: Initialize OpenSearch Client

In [ ]:
# Create OpenSearch client
client = get_os_client()
print("✅ OpenSearch client initialized")

# Configure cluster for OpenAI
configure_cluster_for_openai(client)
print("✅ Cluster configured")

## 🔢 Step 2: Setup Embedding Model

In [ ]:
print("🔢 Registering text embedding model...\n")

# Register embedding model
embedding_model_body = {
    "name": "huggingface/sentence-transformers/all-MiniLM-L12-v2",
    "version": "1.0.2",
    "model_format": "TORCH_SCRIPT"
}

response = client.transport.perform_request(
    'POST',
    '/_plugins/_ml/models/_register?deploy=true',
    body=embedding_model_body
)

embedding_task_id = response['task_id']
print(f"📝 Task ID: {embedding_task_id}")

# Wait for model
print("\n⏳ Waiting for embedding model...")
while True:
    task_response = client.transport.perform_request('GET', f'/_plugins/_ml/tasks/{embedding_task_id}')
    state = task_response['state']
    print(f"   Status: {state}")
    
    if state == 'COMPLETED':
        embedding_model_id = task_response['model_id']
        print(f"\n✅ Embedding model deployed: {embedding_model_id}")
        break
    elif state == 'FAILED':
        raise Exception("Model deployment failed")
    
    time.sleep(10)

## 🤖 Step 3: Setup OpenAI LLM

In [ ]:
# Create OpenAI connector
connector_id = create_openai_connector(client, model_name="gpt-4o-mini")
print(f"✅ OpenAI connector: {connector_id}")

# Register and deploy OpenAI model
llm_model_id = register_and_deploy_openai_model(client, connector_id, model_name="gpt-4o-mini")
print(f"✅ OpenAI model: {llm_model_id}")

## 📚 Step 4: Create Knowledge Base

In [ ]:
# Create ingest pipeline
pipeline_name = "rag_embedding_pipeline"

pipeline_body = {
    "description": "Text embedding pipeline for RAG",
    "processors": [
        {
            "text_embedding": {
                "model_id": embedding_model_id,
                "field_map": {
                    "text": "text_embedding"
                }
            }
        }
    ]
}

client.ingest.put_pipeline(id=pipeline_name, body=pipeline_body)
print(f"✅ Pipeline created: {pipeline_name}")

In [ ]:
# Create vector index
index_name = "rag_knowledge_base"

# Delete if exists
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

index_body = {
    "settings": {
        "index": {
            "knn": True,
            "default_pipeline": pipeline_name
        }
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "text_embedding": {
                "type": "knn_vector",
                "dimension": 384,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "lucene"
                }
            }
        }
    }
}

client.indices.create(index=index_name, body=index_body)
print(f"✅ Index created: {index_name}")

In [ ]:
# Index knowledge base documents
print("\n📝 Indexing knowledge base...")

knowledge_docs = [
    {
        "text": "Seattle's metro area population in 2023 is 3,519,000, showing a 0.86% increase from 2022 when it was 3,489,000. The city has experienced steady growth over the past decade, with a 0.81% increase from 2021 to 2022."
    },
    {
        "text": "New York City remains the largest metro area in the United States with 18,937,000 people in 2023, representing a 0.37% increase from 2022's population of 18,867,000. Growth has been modest but consistent."
    },
    {
        "text": "Austin, Texas has experienced the fastest growth among major US cities. The metro population reached 2,228,000 in 2023, a remarkable 2.39% increase from 2022. This rapid growth is driven by tech industry expansion and favorable business climate."
    },
    {
        "text": "Chicago's metro area population in 2023 is 8,937,000, with a modest 0.4% increase from 2022. The city ranks as the third-largest metro area in the United States and continues to be a major economic hub."
    },
    {
        "text": "Miami's metro population grew to 6,265,000 in 2023, showing a healthy 0.8% increase from 2022. The city continues to attract residents from across the country due to its warm climate and growing economy."
    },
    {
        "text": "San Francisco Bay Area, including Silicon Valley, has a population of approximately 4.7 million. Known globally as the center of technology innovation, it hosts headquarters of major tech companies like Apple, Google, and Meta."
    },
    {
        "text": "Portland, Oregon's metro population is around 2.5 million. The city is known for its environmental consciousness, craft brewery culture, and strong tech presence with companies like Intel having significant operations there."
    }
]

for i, doc in enumerate(knowledge_docs, 1):
    client.index(index=index_name, id=str(i), body=doc)

client.indices.refresh(index=index_name)
print(f"✅ Indexed {len(knowledge_docs)} documents")

## 🤖 Step 5: Create RAG Agent

In [ ]:
print("\n🤖 Creating RAG agent...")

rag_agent_body = {
    "name": "RAG_Agent_OpenAI",
    "type": "flow",
    "description": "RAG agent using OpenAI GPT for intelligent question answering",
    "tools": [
        {
            "type": "RAGTool",
            "parameters": {
                "embedding_model_id": embedding_model_id,
                "inference_model_id": llm_model_id,
                "index": index_name,
                "embedding_field": "text_embedding",
                "source_field": ["text"],
                "input": "${parameters.question}",
                "doc_size": 3,
                "k": 5,
                "prompt": """\n\nYou are a professional data analyst with expertise in demographics and urban statistics.

Your task:
1. Carefully read the context provided below
2. Answer the question based ONLY on the information in the context
3. If the answer is not in the context, say "I don't have that information"
4. Provide specific numbers and facts when available
5. Be concise and clear

Context:
${parameters.output_field}

Question: ${parameters.question}

Answer:"""
            }
        }
    ]
}

response = client.transport.perform_request(
    'POST',
    '/_plugins/_ml/agents/_register',
    body=rag_agent_body
)

agent_id = response['agent_id']
print(f"✅ RAG Agent created: {agent_id}")

## 🧪 Step 6: Test RAG Agent

Let's ask questions and see how RAG provides accurate, grounded answers!

In [ ]:
# Test questions
test_questions = [
    "What was Seattle's population increase from 2022 to 2023?",
    "Which city had the fastest growth rate in 2023?",
    "Compare the populations of Austin and Chicago in 2023",
    "What factors contributed to Miami's growth?",
    "Which cities are known for technology industry?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"🔍 Question {i}: {question}")
    print('='*70)
    
    response = client.transport.perform_request(
        'POST',
        f'/_plugins/_ml/agents/{agent_id}/_execute',
        body={"parameters": {"question": question}}
    )
    
    # Extract answer from response
    if 'inference_results' in response:
        for result in response['inference_results']:
            if 'output' in result:
                for output in result['output']:
                    if 'result' in output and 'name' in output:
                        if output['name'] == 'response':
                            # This is the LLM's final answer
                            try:
                                answer_data = json.loads(output['result'])
                                # Extract text from OpenAI response format
                                if 'choices' in answer_data:
                                    answer = answer_data['choices'][0]['message']['content']
                                    print(f"\n💡 Answer:\n{answer}")
                                else:
                                    print(f"\n💡 Answer:\n{output['result']}")
                            except:
                                print(f"\n💡 Answer:\n{output['result']}")
    
    time.sleep(2)  # Rate limiting

## 💡 Step 7: Understanding RAG Benefits

### What Just Happened?

1. **Your Question** → Converted to vector
2. **Semantic Search** → Found top 3 most relevant documents
3. **Context Assembly** → Documents passed to OpenAI
4. **LLM Generation** → OpenAI generated answer based on context
5. **Grounded Response** → Answer is factual, not hallucinated

### Key Advantages:

| Aspect | Benefit |
|--------|--------|
| 🎯 **Accuracy** | Answers based on YOUR data, not outdated training data |
| 📚 **Sources** | Can cite specific documents |
| 🔄 **Current** | Always uses latest indexed information |
| 🛡️ **Reliable** | Reduced hallucinations - LLM grounded in facts |
| 🎨 **Customizable** | Tune prompts for your domain |

### Comparison:

**Without RAG (Pure LLM)**:
- ❌ May provide outdated information
- ❌ Could hallucinate facts
- ❌ No access to proprietary data

**With RAG**:
- ✅ Current, accurate information
- ✅ Grounded in retrieved documents
- ✅ Works with your private knowledge base

## 🔧 Step 8: Tuning RAG Performance

### Key Parameters:

```python
{
    "doc_size": 3,      # Number of documents to retrieve
    "k": 5,             # k-NN search parameter
    "prompt": "..."    # System prompt for LLM
}
```

### Optimization Tips:

1. **`doc_size`**: 
   - Too few → May miss relevant context
   - Too many → LLM context limit, slower responses
   - Sweet spot: 2-5 documents

2. **`k`**:
   - Higher k → More candidates, better recall
   - Trade-off with performance
   - Typical: k = 2 * doc_size

3. **`prompt`**:
   - Clear instructions to LLM
   - Define behavior for missing information
   - Include context formatting

4. **Embedding Model**:
   - Choose domain-appropriate model
   - Trade-off: size vs. quality
   - all-MiniLM-L12-v2: Good general purpose

## 🎓 Step 9: Key Takeaways

### What You Learned:

1. **✅ RAG Architecture**: Retrieval → Augmentation → Generation
2. **✅ Component Integration**: Embedding model + Vector search + LLM
3. **✅ Knowledge Base**: How to create and index domain-specific data
4. **✅ Prompt Engineering**: Designing effective system prompts for RAG

### Best Practices:

- 🎯 **Chunk Size**: Break documents into appropriate chunks (not too small/large)
- 🎯 **Quality Data**: Index clean, well-structured documents
- 🎯 **Clear Prompts**: Instruct LLM how to use context
- 🎯 **Monitor Results**: Track retrieval quality and answer accuracy
- 🎯 **Iterate**: Test different doc_size and k values

### RAG Use Cases:

- 📚 **Documentation Q&A**: Answer questions about your docs
- 🏢 **Enterprise Search**: Query internal knowledge bases
- 🎓 **Education**: Tutoring based on course materials
- 🏥 **Healthcare**: Medical information from literature
- ⚖️ **Legal**: Search case law and regulations

### Next Steps:

- 📘 Combine RAG with **ConversationalAgent** for chat interfaces
- 📘 Explore **NeuralSparseSearchTool** for alternative retrieval
- 📘 Implement **hybrid search** (keyword + semantic)
- 📘 Add **re-ranking** for better result quality

---

## 🧹 Cleanup (Optional)

In [ ]:
# Uncomment to clean up resources
# from agent_helpers import cleanup_resources

# cleanup_resources(
#     client=client,
#     model_ids=[embedding_model_id, llm_model_id],
#     agent_ids=[agent_id],
#     index_names=[index_name]
# )

# # Delete pipeline and connector
# client.ingest.delete_pipeline(id=pipeline_name)
# client.transport.perform_request('DELETE', f'/_plugins/_ml/connectors/{connector_id}')

# print("\n✅ Cleanup completed")

print("\n📝 Note: Uncomment the code above to clean up resources after the demo")